**Create a model_schema - The unified dataframe must follow this schema**

In [1]:
from pyspark.sql import types as T
from pyspark.sql import DataFrame

# gather in on central schema relevant data from three providers
model_schema = T.StructType([
    T.StructField("title", T.StringType(), True),
    T.StructField("year", T.IntegerType(), True),
    T.StructField("critic_score_pct", T.DoubleType(), True),
    T.StructField("top_critic_score", T.DoubleType(), True),
    T.StructField("total_critic_reviews", T.LongType(),True),
    T.StructField("audience_avg_score", T.DoubleType(), True),
    T.StructField("total_audience_ratings", T.LongType(), True),
    T.StructField("domestic_gross_usd", T.LongType(), True),
    T.StructField("international_gross_usd", T.LongType(), True),
    T.StructField("production_budget_usd", T.LongType(), True),
    T.StructField("marketing_spend_usd", T.LongType(), True)
])

model_columns = [f.name for f in model_schema.fields]

StatementMeta(, 44d7f58a-eca0-424e-a298-8d32c96d0eb3, 3, Finished, Available, Finished, False)

**Helper functions**
1. Clean_title is used to clean movie's title and use it later on as a merge key
2. enforce_schema => enforces schema for any dataframe that is created from reading each providers source

In [2]:
from pyspark.sql import functions as F

def clean_title(col_name: str) -> F.Column:
    """ Clean title, convert to lower, remove white spaces at beggining or end, remove extra white spaces """
    return F.regexp_replace(F.trim(F.lower(F.col(col_name))),r"\s+"," ")


def enforce_schema(df: DataFrame) -> DataFrame:
    """missing columns are added as null"""
    existingColumns = set(df.columns)

    for field in model_schema.fields:
        if field.name not in existingColumns:
            df = df.withColumn(field.name, F.lit(None).cast(field.dataType)) #add a null
        else:
            df = df.withColumn(field.name, F.col(field.name).cast(field.dataType))

    return df.select(model_columns)

StatementMeta(, 44d7f58a-eca0-424e-a298-8d32c96d0eb3, 4, Finished, Available, Finished, False)

**Base Provider. Acts as a contract for the actual providers**
- Each provider must implement extract and transform methods
- In case of adding a new provider, it is as easy as implementing this contract

In [3]:
import abc

class BaseProvider(abc.ABC):
    
    @abc.abstractmethod
    def extract(self) -> DataFrame:
        """Ingest data from the sources (csv, json files) into a Spark DataFrame."""

    @abc.abstractmethod
    def transform(self, raw: DataFrame) -> DataFrame:
        """Rename / cast columns to be unified following the model schema."""

    def extract_transform(self) -> DataFrame:
        return self.transform(self.extract())

StatementMeta(, 44d7f58a-eca0-424e-a298-8d32c96d0eb3, 5, Finished, Available, Finished, False)

**Define schemas as constants**
If schema changes or a new provider is added, then update this cell

In [4]:
from pyspark.sql import types as T

# Read schema from CSV of first Provider
P1_SCHEMA = T.StructType([
    T.StructField("movie_title", T.StringType(), True),
    T.StructField("release_year", T.IntegerType(), True),
    T.StructField("critic_score_percentage", T.DoubleType(),True),
    T.StructField("top_critic_score", T.DoubleType(),True),
    T.StructField("total_critic_reviews_counted", T.DoubleType(),True),
])

# Read schema from json file of second Provider
P2_SCHEMA = T.StructType([
    T.StructField("title", T.StringType(), True),
    T.StructField("year", T.StringType(), True), 
    T.StructField("audience_average_score", T.DoubleType(), True),
    T.StructField("total_audience_ratings", T.LongType(), True),
    T.StructField("domestic_box_office_gross", T.LongType(), True),
])

# Read schema of third provider files (Internationa and Domestic gross)
P3_SCHEMA = T.StructType([
    T.StructField("film_name", T.StringType(), True),
    T.StructField("year_of_release", T.IntegerType(), True),
    T.StructField("box_office_gross_usd",T.LongType(),True),
])

# Read schema of third provider files (Financial)
P3_FINANCIAL_SCHEMA = T.StructType([
    T.StructField("film_name", T.StringType(), True),
    T.StructField("year_of_release", T.IntegerType(), True),
    T.StructField("production_budget_usd", T.LongType(), True),
    T.StructField("marketing_spend_usd", T.LongType(), True),
])

StatementMeta(, 44d7f58a-eca0-424e-a298-8d32c96d0eb3, 6, Finished, Available, Finished, False)

**Ingest Data from First Provider (provider1.csv)**

In [5]:
class CriticProvider(BaseProvider):
    def __init__(self, filepath:str):
        self.filepath = filepath

    def extract(self) -> DataFrame:        
        return (spark.read.option("header",True).schema(P1_SCHEMA).csv(self.filepath))

    def transform(self, raw: DataFrame) -> DataFrame:
        #basically rename columns according to model schema
        df = (raw.filter(F.col("movie_title").isNotNull() & F.col("release_year").isNotNull())
        .withColumnRenamed("movie_title","title")
        .withColumnRenamed("release_year", "year")
        .withColumnRenamed("critic_score_percentage", "critic_score_pct")
        .withColumnRenamed("total_critic_reviews_counted","total_critic_reviews")
        )
        return enforce_schema(df)

StatementMeta(, 44d7f58a-eca0-424e-a298-8d32c96d0eb3, 7, Finished, Available, Finished, False)

**Ingest Data from Second Provider (json file)**

In [6]:
class AudienceProvider(BaseProvider):

    def __init__(self, filepath: str):
        self.filepath = filepath

    def extract(self) -> DataFrame:
        return (
            spark.read.option("multiLine", True).schema(P2_SCHEMA).json(self.filepath)
        )

    def transform(self, raw: DataFrame) -> DataFrame:
        df = (
            raw
            .filter(F.col("title").isNotNull() & F.col("year").isNotNull())
            .withColumnRenamed("audience_average_score",    "audience_avg_score")
            .withColumnRenamed("domestic_box_office_gross", "domestic_gross_usd")
        )
        return enforce_schema(df)

StatementMeta(, 44d7f58a-eca0-424e-a298-8d32c96d0eb3, 8, Finished, Available, Finished, False)

**Ingest Data from Third Provider (different csv files)**

In [7]:
class BoxOfficeMetricsProvider(BaseProvider):

    def __init__(self, domestic_path: str, international_path: str, financials_path: str):
        self.domestic_path      = domestic_path
        self.international_path = international_path
        self.financials_path    = financials_path

    def _read_csv(self, path: str, schema: T.StructType) -> DataFrame:
        return spark.read.option("header", True).schema(schema).csv(path)

    def extract(self) -> dict:
        return {
            "domestic":      self._read_csv(self.domestic_path, P3_SCHEMA),
            "international": self._read_csv(self.international_path, P3_SCHEMA),
            "financials":    self._read_csv(self.financials_path, P3_FINANCIAL_SCHEMA),
        }

    def transform(self, raw: dict) -> DataFrame:
        #joining key
        key = ["film_name", "year_of_release"]

        domestic = raw["domestic"].withColumnRenamed("box_office_gross_usd", "domestic_gross_usd")
        international = raw["international"].withColumnRenamed("box_office_gross_usd", "international_gross_usd")
        financial = raw["financials"]

        # Include even if a movie is missing from any file
        df = (
            domestic
            .join(international, on=key, how="outer")
            .join(financial, on=key, how="outer")
            .withColumnRenamed("film_name", "title")
            .withColumnRenamed("year_of_release", "year")
        )
        return enforce_schema(df)

StatementMeta(, 44d7f58a-eca0-424e-a298-8d32c96d0eb3, 9, Finished, Available, Finished, False)

**MERGE DATAFRAMES FROM THREE PROVIDERS**
1. for each provider: extract data,transform it and enforce the model_schema
2. Combine the three datasets
3. Mind possible conflicts (coalesce accross rows) so that first provider wins or remains
4. Add calculated (derived) columns
5. Could consider extra columns like Roi or any other extra calculation, currently commented
6. drop unncessary columns

In [8]:
def merge_providers(providers: list[BaseProvider]) -> DataFrame:
    #asign order to solve conflicts. First providers remains
    dataframe_list = [provider.extract_transform().withColumn("_order", F.lit(i)) for i, provider in enumerate(providers)]

    #get the first dataframe from the first source
    combined = dataframe_list[0]

    #combine dataframes into one
    for df in dataframe_list[1:]:
        combined = combined.unionByName(df, allowMissingColumns=True)

    #add extra column to aggregate then by. Use clean title and concat year
    combined = combined.withColumn("merge_key", F.concat_ws("_",clean_title("title"), F.col("year").cast("string")))

    # Sort by order first to solve conflicts, picking the first non-null value per column 
    # equivalent of SQL COALESCE across rows.
    agg_exprs = [F.first(F.col(c), ignorenulls=True).alias(c) for c in model_columns]
    coalesced = (
        combined
        .orderBy("_order")
        .groupBy("merge_key")
        .agg(*agg_exprs)
    )

    # Derived columns, calculated
    unified = (
        coalesced
        .withColumn(
            "worldwide_gross_usd",
            F.col("domestic_gross_usd") + F.col("international_gross_usd")
        )
        .withColumn(
            "total_cost",
            F.coalesce(F.col("production_budget_usd"), F.lit(0)) +
            F.coalesce(F.col("marketing_spend_usd"),   F.lit(0))
        )
        # .withColumn(
        #     "roi",
        #     F.when(
        #         F.col("total_cost") > 0,
        #         F.round(
        #             (F.col("worldwide_gross_usd") - F.col("total_cost")) / F.col("total_cost"),
        #             4
        #         )
        #     )
        # )
        .drop("merge_key", "_order", "total_cost")
        .orderBy("year")
    )

    return unified

StatementMeta(, 44d7f58a-eca0-424e-a298-8d32c96d0eb3, 10, Finished, Available, Finished, False)

**Run the notebook**
1. Set providers list
2. Read, extract and merge dataframes
3. Display Results
4. Perhaps saving to a Delta Table?

In [10]:
filepath = "Files"

# set providers
providers = [
    CriticProvider(f"{filepath}/provider1.csv"),
    AudienceProvider(f"{filepath}/provider2.json"),
    BoxOfficeMetricsProvider(
        f"{filepath}/provider3_domestic.csv",
        f"{filepath}/provider3_international.csv",
        f"{filepath}/provider3_financials.csv",
    ),
]

unified_df = merge_providers(providers)

print(f"Pipeline complete — {unified_df.count()} movies in unified dataset.\n")
#unified_df.show(truncate=False)
display(unified_df)

# from this point perhaps save results to a delta table
unified_df.write.mode("overwrite").saveAsTable("Movies")

StatementMeta(, 44d7f58a-eca0-424e-a298-8d32c96d0eb3, 12, Finished, Available, Finished, False)

Pipeline complete — 3 movies in unified dataset.



SynapseWidget(Synapse.DataFrame, 7b6bbeb6-5d56-4ade-9ad3-c551cc48cd2b)

**Query results from Delta Tabla**

In [ ]:
df = spark.sql("SELECT * FROM ClarityLH.dbo.movies LIMIT 1000")
display(df)

StatementMeta(, 44d7f58a-eca0-424e-a298-8d32c96d0eb3, -1, Cancelled, , Cancelled, True)